# Casa 1. Observabilidad

**Duración orientativa: 30 min**

Objetivo: leer las trazas de V0 a V7, comparar latencia y coste por paso y encontrar el paso más caro.
Las trazas se crearon al ejecutar los notebooks 3 a 7. Este notebook las lee de Langfuse con el SDK;
si no hay Langfuse configurado, las lee de `trazas/trazas.jsonl`.

Referencia: https://langfuse.com/docs

In [1]:
from rag_lab import CFG, set_seeds, summary

# CFG.FAST_MODE = False  # corpus de control completo (por defecto se lee FAST_MODE de .env)
# CFG.DOC_PATH = CFG.DATA_DIR / "mi_documento.pdf"  # cualquier PDF sirve
set_seeds()
summary()

Documento:   /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/data/documento.pdf (existe)
FAST_MODE:   True
Dispositivo: cuda
GEN_MODEL:   google/gemini-3.1-flash-lite
JUDGE_MODEL: openai/gpt-5-mini
VLM_MODEL:   google/gemini-3.1-flash-lite
OpenRouter:  clave configurada


Trazas:      jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl  (alumno: profesor)


/home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/rag_lab/tracing.py:49: UserWarning: No se pudo conectar con Langfuse ([Errno 111] Connection refused): las trazas se guardan en JSONL.
  warnings.warn(f"No se pudo conectar con Langfuse ({e}): las trazas se guardan en JSONL.")


## Cómo se registran las trazas

`rag_lab/tracing.py` registra cada paso decorado con `@traced` (`retrieve`, `rerank`, `generate`, `judge`):
latencia, tokens, coste e ids de los fragmentos. Si hay `LANGFUSE_HOST` y claves en `.env`, las trazas van a
Langfuse con tu `ALUMNO` como usuario; si no, a `trazas/trazas.jsonl`. No hay que configurar nada.

Para mirar una traza: abre la URL de Langfuse, entra en el proyecto, ve a *Tracing* y filtra por *User ID*
igual a tu `ALUMNO`. Cada traza se llama `rag-<versión>` y contiene sus pasos con duración y coste.

In [2]:
from rag_lab import tracing, versions

print("Destino de las trazas:", tracing.backend())
output = versions.build("V1").answer("¿Qué porcentaje de participación tiene Dative en la fase 4?")
print(output["respuesta"])
print("trace_id:", output["trace_id"])
tracing.events_df().tail(2)[["paso", "latencia_ms", "tokens", "coste_usd", "ids_fragmentos"]]

Destino de las trazas: jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl


Dative tiene un 10% de participación en la fase 4 [p. 12, 13].
trace_id: 3a0331040b4940b99f054e21282c3d52


,paso,latencia_ms,tokens,coste_usd,ids_fragmentos
0,retrieve,7.75,0,0.000000,"[cabeceras-0037, cabeceras-0018, cabeceras-000..."
1,generate,1336.41,1779,0.000477,[]


## Trazas de V0 a V7

In [3]:
import json
import os

import pandas as pd
from rag_lab import tracing

VERSIONS = [f"V{i}" for i in range(8)]

def langfuse_observations(version, max_pages=5):
    from langfuse import Langfuse

    lf = Langfuse(public_key=os.environ["LANGFUSE_PUBLIC_KEY"], secret_key=os.environ["LANGFUSE_SECRET_KEY"],
                  base_url=os.environ["LANGFUSE_HOST"])
    rows, cursor = [], None
    for _ in range(max_pages):
        r = lf.api.observations.get_many(user_id=CFG.STUDENT, version=version, limit=100, cursor=cursor,
                                         fields="core,basic,usage,metadata")
        for o in r.data:
            d = o.dict()
            meta = d.get("metadata") or {}
            rows.append({"version": version, "trace_id": d["traceId"], "paso": d["name"], "tipo": d["type"],
                          "latencia_langfuse_ms": (d.get("latency") or 0) * 1000,
                          "latencia_ms": float(meta.get("latencia_ms", 0) or 0),
                          "coste_usd": d.get("totalCost") or 0.0,
                          "tokens": (d.get("usageDetails") or {}).get("total", 0)})
        cursor = r.meta.cursor if r.meta else None
        if not cursor:
            break
    return rows

if tracing.backend().startswith("langfuse"):
    obs = pd.DataFrame([f for v in VERSIONS for f in langfuse_observations(v)])
else:
    with open(tracing.jsonl_path(), encoding="utf-8") as f:
        obs = pd.DataFrame([json.loads(l) for l in f])
    obs = obs[obs["version"].isin(VERSIONS)].assign(latencia_langfuse_ms=obs["latencia_ms"])
obs = obs[obs["paso"] != "responder"]
print(len(obs), "pasos leídos de", tracing.backend())
obs.head()

508 pasos leídos de jsonl /home/mikel/github/2627/iab4/reto-1/semana2-reorganizada/trazas/trazas.jsonl


,fecha,trace_id,version,alumno,paso,latencia_ms,tokens,coste_usd,ids_fragmentos,modelo,pregunta,latencia_langfuse_ms
2,2026-11-02T10:29:33.151190+00:00,1e07681a37e04091a0e20264af2cf87e,V1,profesor,retrieve,4.86,0,0.000000,"[cabeceras-0037, cabeceras-0018, cabeceras-000...",NaN,¿Qué porcentaje de participación tiene Dative ...,4.86
3,2026-11-02T10:29:33.152151+00:00,1e07681a37e04091a0e20264af2cf87e,V1,profesor,generate,1336.16,1779,0.000477,[],google/gemini-3.1-flash-lite,¿Qué porcentaje de participación tiene Dative ...,1336.16
4,2026-11-02T10:29:50.889513+00:00,025063060cef469b976e84cfcd88ac4a,V0,profesor,retrieve,5.52,0,0.000000,"[fijo-0003, fijo-0029, fijo-0004, fijo-0061, f...",NaN,¿Qué línea se centra en la promoción del equil...,5.52
5,2026-11-02T10:29:50.890416+00:00,025063060cef469b976e84cfcd88ac4a,V0,profesor,generate,1436.34,904,0.000271,[],google/gemini-3.1-flash-lite,¿Qué línea se centra en la promoción del equil...,1436.34
7,2026-11-02T10:29:50.896041+00:00,427dc9cd01c34174a617386d51df5c6f,V0,profesor,retrieve,4.73,0,0.000000,"[fijo-0008, fijo-0105, fijo-0018, fijo-0046, f...",NaN,Cuál es la duración total del proyecto según e...,4.73


Dos latencias distintas: `latencia_langfuse_ms` es lo que duró el paso en esa ejecución (casi 0 si la llamada
al LLM vino de la caché) y `latencia_ms` es la que registra `tracing.py`, que suma la latencia original de la
llamada al LLM. Para comparar versiones usamos la segunda.

In [4]:
obs.pivot_table(index="version", columns="paso", values="latencia_ms", aggfunc="median").round(0)

paso,generate,query_transform,rerank,retrieve
version,,,,
V0,747.0,NaN,NaN,3.0
V1,764.0,NaN,NaN,3.0
V2,653.0,NaN,NaN,1.0
V3,633.0,NaN,NaN,4.0
V4,732.0,NaN,1161.0,6.0
V5,765.0,NaN,1536.0,7.0
V6,725.0,850.0,1442.0,20.0
V7,947.0,850.0,1605.0,40.0


In [5]:
by_trace = obs.groupby(["version", "trace_id"]).agg(coste_usd=("coste_usd", "sum"), latencia_ms=("latencia_ms", "sum"))
by_trace.groupby("version").median().round(5)

,coste_usd,latencia_ms
version,,
V0,0.00027,750.405
V1,0.00041,767.420
V2,0.00035,654.015
V3,0.00042,637.575
V4,0.00042,1907.240
V5,0.00052,2366.005
V6,0.00066,3089.080
V7,0.00092,3490.145


In [6]:
costliest_step = (obs.groupby(["version", "paso"])["latencia_ms"].median().reset_index()
             .sort_values("latencia_ms", ascending=False).groupby("version").head(1).set_index("version"))
costliest_step.round(0)

,paso,latencia_ms
version,,
V7,rerank,1605.0
V5,rerank,1536.0
V6,rerank,1442.0
V4,rerank,1161.0
V1,generate,764.0
V0,generate,747.0
V2,generate,653.0
V3,generate,633.0


Prueba: abre Langfuse, filtra por tu usuario y por la versión V7, abre la traza más lenta y comprueba qué paso
ocupa más tiempo. ¿Coincide con la tabla anterior? ¿Qué paso añadió cada versión respecto a la anterior?